# 01 · Prepare Hermes function-calling data for Soup

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` · **Dataset:** `NousResearch/hermes-function-calling-v1`

Import this notebook into Kaggle, enable **Internet**, and run cells in order.
Preparation runs on CPU; select **Accelerator: None** to preserve T4 quota.
It also works in a T4 notebook, without allocating model weights on the GPU.
Use a fresh session with Python 3.11–3.13.

This notebook downloads pinned data and tokenizer files, audits references,
creates reproducible family-separated splits, checks actual Soup loss masks,
runs Soup's data diagnostics, and exports one ZIP for the training notebook.
**It never trains a model, loads model weights, executes tools, or calls an LLM.**

The earlier smoke run used 16 examples. This notebook targets **1,600 training,
150 validation and 150 test examples**, with a hard minimum of **1,000 training**.
Actual counts are determined when you run it. Malformed labels are excluded,
never repaired to reach a quota. If the primary file is insufficient, it audits
the Glaive file **inside the same pinned Hermes repository**. If the resulting
pool is insufficient, it writes the audit and stops before producing a ready bundle.

Outputs go to a unique directory under `/kaggle/working/hermes-soup-data/`.
No Kaggle API key or Hugging Face token is required for these public downloads.

## 1. Freeze the preparation policy

Change these settings before the first run, then retain the policy with your
experiment. Sequence lengths are candidates for complete-example retention,
**not claims of measured GPU fit**. Only length 512, microbatch 1 has been
tested in the earlier T4 smoke experiment. Larger lengths need GPU calibration
in the training notebook. Existing output directories are never overwritten.

In [1]:
import collections
import datetime as dt
from decimal import Decimal
import hashlib
import importlib.metadata as metadata
import json
import math
import os
from pathlib import Path
import re
import subprocess
import sys
import time
import unicodedata
import zipfile

SOUP_REV = "4cff43bf967e96a484381205e5125e2fc39227f1"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REV = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
DATA_ID = "NousResearch/hermes-function-calling-v1"
DATA_REV = "dae3e1d28cfbcf4b915c04ea1e072030529b4bda"
PRIMARY = "func-calling-singleturn.json"
EXPANSION = "glaive-function-calling-5k.json"
PRIMARY_SHA256 = "fb2df9fe3f295dfd65b223c2dbf04d6b90006357d737f8f8927749f0bea09527"
SEED = 42
TRAIN_TARGET, TRAIN_MIN, VAL_TARGET, TEST_TARGET = 1600, 1000, 150, 150
LENGTH_CANDIDATES = (512, 1024, 1536, 2048)
ALLOW_HERMES_GLAIVE_EXPANSION = True
assert 1000 <= TRAIN_MIN <= TRAIN_TARGET <= 2000
assert VAL_TARGET > 0 and TEST_TARGET > 0
assert tuple(sorted(set(LENGTH_CANDIDATES))) == LENGTH_CANDIDATES
assert all(type(n) is int and 0 < n <= 2048 for n in LENGTH_CANDIDATES)
assert (3, 11) <= sys.version_info[:2] < (3, 14), "Use Python 3.11–3.13."

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = Path("/kaggle/working/hermes-soup-data") / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
os.chdir(OUT)
RAW, PREPARED, AUDIT = [OUT / name for name in ("raw", "data/prepared", "audit")]
for folder in (RAW, PREPARED, AUDIT, AUDIT / "logs"):
    folder.mkdir(parents=True, exist_ok=True)

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def digest(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")

def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(canonical(row) + "\n")

POLICY = {
    "soup_commit": SOUP_REV, "model": MODEL_ID, "model_revision": MODEL_REV,
    "dataset": DATA_ID, "dataset_revision": DATA_REV, "seed": SEED,
    "train_target": TRAIN_TARGET, "train_min": TRAIN_MIN,
    "validation_target": VAL_TARGET, "test_target": TEST_TARGET,
    "length_candidates": LENGTH_CANDIDATES,
    "expansion_allowed": ALLOW_HERMES_GLAIVE_EXPANSION,
    "task": "English single-turn call-only generation; multiple calls retain order",
    "reference_policy": "strict JSON and schemas; no label repair; exclude conflicting references",
    "split_policy": "connected components of source ID, normalized request, tool name and structural schema family",
    "selection_policy": "small components reserved for holdouts; seeded round-robin within assigned components",
    "schema_default": "Draft 7 when $schema is absent; declared supported dialect otherwise",
    "schema_limits": "no external references, dynamic/recursive references or unknown string formats",
    "loss_policy": "assistant content and EOT only; role headers and trailing whitespace masked",
}
write_json(AUDIT / "policy.json", POLICY)

def run_command(argv, name, check=True, timeout=600):
    """Write full output directly to disk; keep notebook console output bounded."""
    log = AUDIT / "logs" / f"{name}.log"
    record = {"argv": [str(x) for x in argv], "cwd": str(OUT),
              "started_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "log": str(log.relative_to(OUT))}
    start = time.monotonic()
    try:
        with log.open("w", encoding="utf-8") as handle:
            result = subprocess.run(record["argv"], stdout=handle, stderr=subprocess.STDOUT,
                                    timeout=timeout, check=False, cwd=OUT)
        record["exit_code"] = result.returncode
    except subprocess.TimeoutExpired:
        record["timeout_seconds"] = timeout
        record["exit_code"] = None
        raise
    finally:
        record["elapsed_seconds"] = round(time.monotonic() - start, 3)
        record["finished_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
        with (AUDIT / "commands.jsonl").open("a") as handle:
            handle.write(canonical(record) + "\n")
    lines = log.read_text(errors="replace").splitlines()
    print(f"{name}: exit={result.returncode}; {len(lines)} log lines → {log.relative_to(OUT)}")
    if check and result.returncode:
        print("\n".join(lines[-20:]))
        raise RuntimeError(f"{name} failed; inspect its complete log.")
    return result.returncode

print("Output directory:", OUT)
print("Policy SHA-256:", digest(POLICY))

Output directory: /kaggle/working/hermes-soup-data/20261109T210817498784Z
Policy SHA-256: fe43e199521a6b8a177718a80f579c17eeb07f4252d7010ca08ce92874079389


## 2. Install the data tools

Use the inspected Soup commit, the Transformers/TRL versions from the successful
smoke run, and JSON Schema validation with format support. Soup's full training
extra and bitsandbytes are unnecessary for preparing data. Kaggle's CPU PyTorch
path is used only for a small collator check. Shared-image dependency warnings
are saved; a successful data check does not make the whole Kaggle image clean.

In [2]:
run_command([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    f"soup-cli @ git+https://github.com/MakazhanAlpamys/Soup.git@{SOUP_REV}",
    "transformers==5.20.2", "trl==0.29.1", "jsonschema[format-nongpl]>=4.18,<5",
], "install", timeout=900)
pip_check_code = run_command([sys.executable, "-m", "pip", "check"], "pip-check", check=False)
run_command([sys.executable, "-m", "pip", "freeze"], "pip-freeze")

import torch
import yaml
from huggingface_hub import hf_hub_download, snapshot_download
from transformers import AutoTokenizer
from jsonschema import Draft7Validator, FormatChecker, SchemaError, ValidationError, validators
from referencing.exceptions import Unresolvable
from soup_cli.config.loader import load_config_from_string
from soup_cli.data.sft_format import build_format_row
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling

torch.set_num_threads(2)
direct_url = json.loads(metadata.distribution("soup-cli").read_text("direct_url.json") or "{}")
assert direct_url.get("vcs_info", {}).get("commit_id") == SOUP_REV, "Unexpected Soup source identity."
environment = {
    "python": sys.version, "platform": sys.platform, "pip_check_exit_code": pip_check_code,
    "versions": {p: metadata.version(p) for p in ("soup-cli", "torch", "transformers", "trl", "huggingface-hub", "jsonschema")},
    "soup_install": direct_url,
}
write_json(AUDIT / "environment.json", environment)
print(json.dumps(environment["versions"], indent=2))

install: exit=0; 110 log lines → audit/logs/install.log
pip-check: exit=1; 8 log lines → audit/logs/pip-check.log
pip-freeze: exit=0; 861 log lines → audit/logs/pip-freeze.log
{
  "soup-cli": "0.77.2",
  "torch": "2.11.0+cpu",
  "transformers": "5.20.2",
  "trl": "0.29.1",
  "huggingface-hub": "1.33.0",
  "jsonschema": "4.26.0"
}


## 3. Strict parsers and schema checks

A target must be a complete sequence of `<tool_call>` JSON objects. Duplicate
keys, Python literals, non-finite numbers, extra prose, unknown tools and invalid
arguments are excluded. Nothing uses `eval`, broad replacements or generated
repairs. JSON Schema formats are checked; external references are never fetched.
Successful structural validation still does not prove that a label matches the
user's intent. A fixed training/validation review sample is exported later.

In [3]:
class Rejected(ValueError):
    def __init__(self, reason, detail):
        super().__init__(detail)
        self.reason, self.detail = reason, detail

def strict_json(text):
    def pairs(items):
        result = {}
        for key, value in items:
            if key in result:
                raise ValueError(f"duplicate object key: {key}")
            result[key] = value
        return result

    def finite_float(value):
        result = float(value)
        if not math.isfinite(result):
            raise ValueError("non-finite number")
        if Decimal(value) != Decimal(str(result)):
            raise ValueError("JSON number would lose decimal precision when normalized")
        return result

    def invalid_constant(value):
        raise ValueError(f"nonstandard JSON constant: {value}")

    return json.loads(text, object_pairs_hook=pairs, parse_float=finite_float,
                      parse_constant=invalid_constant)

CALL_BLOCK = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)

def parse_calls(text):
    if not isinstance(text, str):
        raise ValueError("assistant content is not a string")
    calls, position = [], 0
    while position < len(text):
        while position < len(text) and text[position].isspace():
            position += 1
        if position == len(text):
            break
        match = CALL_BLOCK.match(text, position)
        if match is None:
            raise ValueError(f"unexpected text or incomplete call at character {position}")
        call = strict_json(match.group(1))
        if not isinstance(call, dict) or set(call) != {"name", "arguments"}:
            raise ValueError("a call must contain exactly name and arguments")
        if not isinstance(call["name"], str) or not call["name"] or not isinstance(call["arguments"], dict):
            raise ValueError("invalid function name or arguments object")
        calls.append(call)
        position = match.end()
    if not calls:
        raise ValueError("no call-only target; abstentions and clarifications require a separate policy")
    return calls

def render_calls(calls):
    return "\n".join("<tool_call>\n" + canonical(call) + "\n</tool_call>" for call in calls)

FORMAT_CHECKER = FormatChecker()
SCHEMA_DIALECTS = {
    "http://json-schema.org/draft-04/schema#", "http://json-schema.org/draft-06/schema#",
    "http://json-schema.org/draft-07/schema#", "https://json-schema.org/draft/2019-09/schema",
    "https://json-schema.org/draft/2020-12/schema",
}
VALIDATOR_CACHE = {}

def schema_validator(schema):
    key = canonical(schema)
    if key in VALIDATOR_CACHE:
        return VALIDATOR_CACHE[key]
    if not isinstance(schema, dict) or schema.get("type") != "object":
        raise ValueError("function parameters must declare an object schema")
    declared = schema.get("$schema")
    if declared is not None and declared not in SCHEMA_DIALECTS:
        raise ValueError(f"unsupported declared schema dialect: {declared}")

    def inspect_schema(value):
        if isinstance(value, dict):
            if "$dynamicRef" in value or "$recursiveRef" in value or "nullable" in value:
                raise ValueError("unsupported dynamic/recursive reference or OpenAPI nullable keyword")
            if "$ref" in value and (not isinstance(value["$ref"], str) or not value["$ref"].startswith("#")):
                raise ValueError("external schema references are disabled")
            if "format" in value and value["format"] not in FORMAT_CHECKER.checkers:
                raise ValueError(f"unknown schema format: {value['format']}")
            for child in value.values():
                inspect_schema(child)
        elif isinstance(value, list):
            for child in value:
                inspect_schema(child)

    inspect_schema(schema)
    validator_cls = validators.validator_for(schema, default=Draft7Validator)
    validator_cls.check_schema(schema)
    validator = validator_cls(schema, format_checker=FORMAT_CHECKER)
    VALIDATOR_CACHE[key] = validator
    return validator

def parse_tools(text):
    tools = strict_json(text)
    if not isinstance(tools, list) or not tools:
        raise ValueError("tools must be a nonempty list")
    functions = {}
    for tool in tools:
        if not isinstance(tool, dict) or tool.get("type") != "function" or not isinstance(tool.get("function"), dict):
            raise ValueError("unsupported tool record")
        function = tool["function"]
        name = function.get("name")
        if not isinstance(name, str) or not name.strip() or name in functions:
            raise ValueError("missing or duplicate tool name")
        schema_validator(function.get("parameters"))
        functions[name] = function
    return tools, functions

def normalized_request(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())

def schema_structure(value):
    if isinstance(value, dict):
        return {k: schema_structure(v) for k, v in value.items()
                if k not in {"description", "title", "examples", "default", "$comment"}}
    if isinstance(value, list):
        return [schema_structure(v) for v in value]
    return value

# Small fixtures run when YOU execute this cell; they do not inspect real test answers.
valid = '<tool_call>{"name":"f","arguments":{"enabled":true}}</tool_call>'
assert len(parse_calls(valid + "\n" + valid)) == 2
assert parse_calls(valid)[0]["arguments"]["enabled"] is True
for bad in (
    '<tool_call>{"name":"f","name":"g","arguments":{}}</tool_call>',
    '<tool_call>{"name":"f","arguments":{"x":NaN}}</tool_call>',
    '<tool_call>{"name":"f","arguments":{"x":1e309}}</tool_call>',
    '<tool_call>{"name":"f","arguments":{"x":1.0000000000000001}}</tool_call>',
    '<tool_call>{"name":"f","arguments":[]}</tool_call>',
    valid + " trailing prose", '<tool_call>{"name":"f","arguments":{}}',
):
    try:
        parse_calls(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Parser accepted a negative fixture")
fixture = schema_validator({"type": "object", "properties": {"n": {"type": "integer"}}, "required": ["n"]})
assert not list(fixture.iter_errors({"n": 2})) and list(fixture.iter_errors({"n": True}))
try:
    schema_validator({"type": "object", "$ref": "https://example.invalid/schema"})
except ValueError:
    pass
else:
    raise AssertionError("External-reference fixture was accepted")
print("Strict JSON, call parsing and schema fixtures passed.")

Strict JSON, call parsing and schema fixtures passed.


## 4. Download the pinned tokenizer and define exact masks

Only tokenizer/config files are downloaded—no `.safetensors` or model weights.
The strict template below applies to this notebook's `system → user → assistant`
messages with tool definitions already in the system content. Tools are **not**
passed a second time through Qwen's `tools=` template argument.

Every candidate must match the native Qwen rendering and token IDs for both
training and inference. Generation markers cover assistant content plus EOT;
they do not change the rendered prompt. The same template is exported in the
tokenizer and Soup data settings. Reverting to the native unmarked template
during training would reintroduce the smoke run's heuristic mask fallback.

In [4]:
native_dir = OUT / "tokenizer-native"
snapshot_download(
    MODEL_ID, revision=MODEL_REV, local_dir=str(native_dir),
    allow_patterns=["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
                    "vocab.json", "merges.txt", "added_tokens.json", "chat_template.jinja", "config.json"],
)
assert not list(native_dir.glob("*.safetensors")) and not list(native_dir.glob("*.bin"))
tokenizer = AutoTokenizer.from_pretrained(native_dir, use_fast=True, trust_remote_code=False, local_files_only=True)
assert tokenizer.is_fast and tokenizer.eos_token == "<|im_end|>"
NATIVE_TEMPLATE = tokenizer.chat_template
assert isinstance(NATIVE_TEMPLATE, str) and NATIVE_TEMPLATE
STRICT_TEMPLATE = (
    "{% for m in messages %}{{ '<|im_start|>' + m['role'] + '\n' }}"
    "{% if m['role'] == 'assistant' %}{% generation %}"
    "{{ m['content'] + '<|im_end|>' }}{% endgeneration %}{{ '\n' }}"
    "{% else %}{{ m['content'] + '<|im_end|>\n' }}{% endif %}{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
(AUDIT / "native-chat-template.jinja").write_text(NATIVE_TEMPLATE)
(OUT / "chat-template.jinja").write_text(STRICT_TEMPLATE)
tokenizer.chat_template = STRICT_TEMPLATE
tokenizer.padding_side = "right"
tokenizer.save_pretrained(OUT / "tokenizer")
SPECIAL_TEXT = tuple(token for token in tokenizer.all_special_tokens if token)

def soup_config(max_length):
    return load_config_from_string(yaml.safe_dump({
        "base": MODEL_ID, "task": "sft", "backend": "transformers", "output": "./artifacts/adapter",
        "data": {"train": "./data/prepared/train.jsonl", "format": "chatml", "val_split": 0,
                 "max_length": max_length, "train_on_responses_only": True, "chat_template": STRICT_TEMPLATE},
        "training": {"train_on_eot": True, "packing": False},
    }))

cfg_audit = soup_config(max(LENGTH_CANDIDATES))
format_row = build_format_row(tokenizer, cfg_audit.data, training_cfg=cfg_audit.training)

def audit_tokens(messages):
    for prefix, generation in ((messages, False), (messages[:-1], True)):
        original = tokenizer.apply_chat_template(prefix, chat_template=NATIVE_TEMPLATE,
            tokenize=False, add_generation_prompt=generation)
        marked = tokenizer.apply_chat_template(prefix, chat_template=STRICT_TEMPLATE,
            tokenize=False, add_generation_prompt=generation)
        assert original == marked, "Strict template changed native rendered text."
        assert tokenizer.apply_chat_template(prefix, chat_template=NATIVE_TEMPLATE,
            tokenize=True, add_generation_prompt=generation, return_dict=False) == tokenizer.apply_chat_template(
            prefix, chat_template=STRICT_TEMPLATE, tokenize=True, add_generation_prompt=generation, return_dict=False)
    full = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False, return_dict=False)
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True, return_dict=False)
    assert full[:len(prompt)] == prompt, "Inference prompt is not a stable token prefix."
    target_eot = full.index(tokenizer.eos_token_id, len(prompt))
    metrics = {"total_tokens": len(full), "prompt_tokens": len(prompt),
               "target_tokens_with_eot": target_eot - len(prompt) + 1}
    if len(full) > max(LENGTH_CANDIDATES):
        return metrics
    encoded = format_row({"messages": messages})
    assert encoded["input_ids"] == full, "Unexpected Soup truncation or token drift."
    assert encoded["attention_mask"] == [1] * len(full)
    expected = [-100] * len(prompt) + full[len(prompt):target_eot + 1] + [-100] * (len(full) - target_eot - 1)
    assert encoded["labels"] == expected, "Soup labels differ from the exact content/EOT policy."
    preferred = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False,
        return_assistant_tokens_mask=True, return_dict=True)
    assert preferred.get("assistant_masks") == [int(label != -100) for label in expected], "Assistant mask path unavailable."
    trained = [label for label in encoded["labels"] if label != -100]
    assert tokenizer.decode(trained, skip_special_tokens=False, clean_up_tokenization_spaces=False) == messages[-1]["content"] + tokenizer.eos_token
    assert any(label != -100 for label in encoded["labels"][1:])
    metrics["shifted_supervised_tokens"] = sum(label != -100 for label in encoded["labels"][1:])
    metrics["supervised_fraction"] = metrics["shifted_supervised_tokens"] / max(1, len(full) - 1)
    return metrics

synthetic = [{"role": "system", "content": "Use the supplied tools."},
             {"role": "user", "content": "Call f with x=1."},
             {"role": "assistant", "content": '<tool_call>\n{"name":"f","arguments":{"x":1}}\n</tool_call>'}]
audit_tokens(synthetic)
print("Native/strict template equivalence and exact Soup mask fixture passed.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Native/strict template equivalence and exact Soup mask fixture passed.


## 5. Audit source rows without modifying references

The primary file is checked against the SHA-256 observed during planning.
Each downloaded file also receives a hash, size and row-count record.
Glaive conversations contribute **only their first system/user/assistant
prefix**; later turns never become independent examples. The paired
`func-calling.json` file is deliberately unused because it shares primary IDs.

Source prompts sometimes mention empty `<tools></tools>` tags in their
instructions. The audit ignores these empty mentions and requires exactly one
nonempty tools block matching the separate tools field. The original system
and user text are retained; only valid assistant JSON is serialized canonically.

In [5]:
source_manifest, source_candidates, raw_rejections = [], [], []
hf_hub_download(DATA_ID, "README.md", repo_type="dataset", revision=DATA_REV, local_dir=str(RAW))

def rejection(row, filename, index, reason, detail):
    return {"source_file": filename, "source_index": index,
            "source_id": row.get("id") if isinstance(row, dict) else None,
            "reason": reason, "detail": str(detail)[:1000], "raw_row_sha256": digest(row)}

def audit_source(filename):
    path = Path(hf_hub_download(DATA_ID, filename, repo_type="dataset", revision=DATA_REV, local_dir=str(RAW)))
    sha = file_sha(path)
    if filename == PRIMARY:
        assert sha == PRIMARY_SHA256, "Pinned primary file does not match the expected bytes."
    rows = strict_json(path.read_text(encoding="utf-8"))
    assert isinstance(rows, list), "Expected the pinned source to be a JSON array."
    source_manifest.append({"file": filename, "revision": DATA_REV, "sha256": sha,
                            "bytes": path.stat().st_size, "rows": len(rows)})
    accepted, excluded = [], []
    for index, row in enumerate(rows):
        try:
            if not isinstance(row, dict) or not isinstance(row.get("id"), str) or not row["id"]:
                raise Rejected("source_structure", "missing source ID or non-object row")
            conversation = row.get("conversations")
            if not isinstance(conversation, list) or len(conversation) < 3:
                raise Rejected("source_structure", "missing first turn")
            if filename == PRIMARY and len(conversation) != 3:
                raise Rejected("source_structure", "primary row is not single-turn")
            prefix = conversation[:3]
            if [m.get("from") if isinstance(m, dict) else None for m in prefix] != ["system", "human", "gpt"]:
                raise Rejected("role_sequence", "expected system/human/gpt first prefix")
            if any(not isinstance(m.get("value"), str) or not m["value"].strip() for m in prefix):
                raise Rejected("empty_content", "empty or non-string message")
            if any(token in m["value"] for token in SPECIAL_TEXT for m in prefix):
                raise Rejected("special_token_in_content", "raw message contains a reserved tokenizer token")
            try:
                tools, functions = parse_tools(row.get("tools"))
            except (ValueError, TypeError, SchemaError) as exc:
                raise Rejected("tools_or_schema", str(exc)) from exc
            blocks = [block.strip() for block in re.findall(r"<tools>(.*?)</tools>", prefix[0]["value"], flags=re.S) if block.strip()]
            if len(blocks) != 1:
                raise Rejected("embedded_tools", "expected exactly one nonempty tools block")
            try:
                embedded = strict_json(blocks[0])
            except (TypeError, ValueError) as exc:
                raise Rejected("embedded_tools", str(exc)) from exc
            if canonical(embedded) != canonical(tools):
                raise Rejected("tools_mismatch", "system and separate tools definitions differ")
            try:
                calls = parse_calls(prefix[2]["value"])
            except ValueError as exc:
                raise Rejected("call_syntax", str(exc)) from exc
            for call in calls:
                if call["name"] not in functions:
                    raise Rejected("unknown_tool", call["name"])
                try:
                    schema_validator(functions[call["name"]]["parameters"]).validate(call["arguments"])
                except ValidationError as exc:
                    raise Rejected("argument_schema", f"{list(exc.absolute_path)}: {exc.message}") from exc
                except Unresolvable as exc:
                    # Fail closed on unsupported/unresolvable schemas; never repair them.
                    raise Rejected("schema_resolution", f"{type(exc).__name__}: {exc}") from exc
            target = render_calls(calls)
            assert canonical(parse_calls(target)) == canonical(calls)
            messages = [{"role": "system", "content": prefix[0]["value"]},
                        {"role": "user", "content": prefix[1]["value"]},
                        {"role": "assistant", "content": target}]
            ordered_tools = sorted(tools, key=lambda tool: tool["function"]["name"])
            item = {
                "id": filename + ":" + row["id"], "source_id": row["id"], "source_file": filename,
                "source_index": index, "raw_row_sha256": digest(row), "messages": messages,
                "tools": tools, "calls": calls, "call_count": len(calls),
                "category": row.get("category", "unknown"), "task": row.get("task", "unknown"),
                "raw_target": prefix[2]["value"], "target_sha256": digest(target),
                "request_key": digest(normalized_request(prefix[1]["value"])),
                "input_key": digest([normalized_request(prefix[1]["value"]), ordered_tools]),
                "tool_names": sorted(functions),
                "schema_keys": sorted(digest([name.casefold(), schema_structure(function["parameters"])]) for name, function in functions.items()),
                "lineage": [{"file": filename, "index": index, "source_id": row["id"]}],
            }
            accepted.append(item)
        except Rejected as exc:
            excluded.append(rejection(row, filename, index, exc.reason, exc.detail))
    source_candidates.extend(accepted)
    raw_rejections.extend(excluded)
    write_json(AUDIT / "sources.json", source_manifest)
    write_jsonl(AUDIT / "source-exclusions.jsonl", raw_rejections)
    print(filename, {"source_rows": len(rows), "schema_valid_call_rows": len(accepted), "excluded": len(excluded)})

README.md:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

## 6. Deduplicate, group families, and select a feasible length

Identical normalized requests plus identical available tools are deduplicated.
If those inputs have conflicting call targets, **all variants are excluded**.
Source IDs and duplicate lineage are retained.

Split components connect rows sharing a source ID, normalized request, any
case-folded tool name, or structural schema family. Grouping happens before
length filtering, so an overlength row cannot silently break a family bridge.
This is stronger than a random row split; it is still not a semantic near-duplicate
detector or proof that all natural-language templates are independent.

Smaller components supply the holdouts while preserving at least 1,000 training
rows. Rows are then selected in deterministic round-robin order across components.
Unused rows remain assigned to their component's split and cannot enter another
split. This selection biases heldouts toward less frequent families; the manifest
records the exact composition. A giant component is reported rather than broken
just to meet counts.

In [6]:
def seeded_key(value):
    return digest([SEED, value])

def deduplicate(items):
    buckets = collections.defaultdict(list)
    for item in items:
        buckets[item["input_key"]].append(item)
    kept, excluded = [], []
    for key in sorted(buckets):
        bucket = sorted(buckets[key], key=lambda x: (x["source_file"] != PRIMARY, x["id"], x["source_index"]))
        conflict = len({canonical(item["calls"]) for item in bucket}) != 1
        if conflict:
            for item in bucket:
                excluded.append({"id": item["id"], "reason": "conflicting_reference", "input_key": key})
            continue
        item = dict(bucket[0])
        item["lineage"] = [entry for other in bucket for entry in other["lineage"]]
        kept.append(item)
        excluded.extend({"id": other["id"], "reason": "duplicate_input_and_target", "kept_id": item["id"]} for other in bucket[1:])
    assert len({item["id"] for item in kept}) == len(kept)
    return kept, excluded

def group_rows(items):
    parent = list(range(len(items)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    owners = {}
    for i, item in enumerate(items):
        keys = [("request", item["request_key"])]
        keys += [("id", entry["source_id"]) for entry in item["lineage"]]
        keys += [("tool", name.casefold()) for name in item["tool_names"]]
        keys += [("schema", key) for key in item["schema_keys"]]
        for key in keys:
            if key in owners:
                a, b = find(i), find(owners[key])
                parent[max(a, b)] = min(a, b)
            else:
                owners[key] = i
    components = collections.defaultdict(list)
    for i, item in enumerate(items):
        components[find(i)].append(item)
    for component in components.values():
        group_id = digest(sorted(item["id"] for item in component))
        for item in component:
            item["group_id"] = group_id
    return items

TOKEN_CACHE = {}
def annotate_tokens(items):
    for i, item in enumerate(items):
        key = digest(item["messages"])
        if key not in TOKEN_CACHE:
            TOKEN_CACHE[key] = audit_tokens(item["messages"])
        item["tokens"] = TOKEN_CACHE[key]
        if (i + 1) % 500 == 0:
            print(f"Audited exact lengths/masks for {i + 1}/{len(items)} rows")
    return items

def choose_rows(groups, limit):
    buckets = [iter(sorted(rows, key=lambda x: (x["source_file"] != PRIMARY, seeded_key(x["id"]))))
               for _, rows in sorted(groups.items(), key=lambda pair: seeded_key(pair[0]))]
    result = []
    while buckets and len(result) < limit:
        active = []
        for bucket in buckets:
            if len(result) == limit:
                break
            item = next(bucket, None)
            if item is not None:
                result.append(item)
                active.append(bucket)
        buckets = active
    return result

def allocate(items, max_length):
    groups = collections.defaultdict(list)
    for item in items:
        if item["tokens"]["total_tokens"] <= max_length:
            groups[item["group_id"]].append(item)
    remaining = dict(groups)
    assigned = {"validation": {}, "test": {}}
    for split, target, other_min in (("validation", VAL_TARGET, TEST_TARGET), ("test", TEST_TARGET, 0)):
        count = 0
        for group_id, rows in sorted(list(remaining.items()), key=lambda pair: (len(pair[1]), seeded_key([split, pair[0]]))):
            if count >= target:
                break
            remaining_count = sum(map(len, remaining.values()))
            if remaining_count - len(rows) < TRAIN_MIN + other_min:
                continue
            assigned[split][group_id] = remaining.pop(group_id)
            count += len(rows)
        if count < target:
            return None
    if sum(map(len, remaining.values())) < TRAIN_MIN:
        return None
    assigned["train"] = remaining
    splits = {name: choose_rows(assigned[name], target) for name, target in
              (("train", TRAIN_TARGET), ("validation", VAL_TARGET), ("test", TEST_TARGET))}
    assignments = {group_id: name for name, group_map in assigned.items() for group_id in group_map}
    return splits, assignments

def plan_pool():
    unique, duplicate_exclusions = deduplicate(source_candidates)
    items = annotate_tokens(group_rows(unique))
    retention, plan = [], None
    for length in LENGTH_CANDIDATES:
        proposed = allocate(items, length)
        retention.append({"max_length": length,
            "retained_rows": sum(x["tokens"]["total_tokens"] <= length for x in items),
            "feasible_grouped_split": proposed is not None})
        if proposed is not None and plan is None:
            plan = (length, *proposed)
    return items, duplicate_exclusions, retention, plan

# Determinism and transitive family grouping fixture.
fixture_rows = [
    {"id": "a", "request_key": "r1", "lineage": [{"source_id": "a"}], "tool_names": ["f"], "schema_keys": []},
    {"id": "b", "request_key": "r2", "lineage": [{"source_id": "b"}], "tool_names": ["f", "g"], "schema_keys": []},
    {"id": "c", "request_key": "r3", "lineage": [{"source_id": "c"}], "tool_names": ["g"], "schema_keys": []},
    {"id": "d", "request_key": "r4", "lineage": [{"source_id": "d"}], "tool_names": ["h"], "schema_keys": []},
]
group_rows(fixture_rows)
assert fixture_rows[0]["group_id"] == fixture_rows[2]["group_id"] != fixture_rows[3]["group_id"]

duplicate_a = {"id": "a", "source_file": PRIMARY, "source_index": 0, "input_key": "same",
               "calls": [{"name": "f", "arguments": {"x": 1}}], "lineage": [{"source_id": "a"}]}
duplicate_b = {**duplicate_a, "id": "b", "source_index": 1, "lineage": [{"source_id": "b"}]}
retained, removed = deduplicate([duplicate_a, duplicate_b])
assert len(retained) == 1 and len(removed) == 1 and len(retained[0]["lineage"]) == 2
conflicting_b = {**duplicate_b, "calls": [{"name": "f", "arguments": {"x": 2}}]}
retained, removed = deduplicate([duplicate_a, conflicting_b])
assert not retained and len(removed) == 2 and all(x["reason"] == "conflicting_reference" for x in removed)

fixture_pool = [{"id": f"row-{i}", "group_id": f"group-{i}", "source_file": PRIMARY,
                 "tokens": {"total_tokens": min(100, LENGTH_CANDIDATES[0])}} for i in range(TRAIN_MIN + VAL_TARGET + TEST_TARGET)]
fixture_plan = allocate(fixture_pool, LENGTH_CANDIDATES[0])
assert fixture_plan is not None and len(fixture_plan[0]["train"]) == TRAIN_MIN
assert canonical(fixture_plan) == canonical(allocate(fixture_pool, LENGTH_CANDIDATES[0]))
fixture_sets = [set(row["id"] for row in fixture_plan[0][name]) for name in ("train", "validation", "test")]
assert not (fixture_sets[0] & fixture_sets[1] or fixture_sets[0] & fixture_sets[2] or fixture_sets[1] & fixture_sets[2])
assert allocate([{**row, "group_id": "one-giant-family"} for row in fixture_pool], LENGTH_CANDIDATES[0]) is None
print("Deduplication, conflicting-reference, transitive grouping and deterministic split fixtures passed.")

audit_source(PRIMARY)
items, dedup_exclusions, retention, plan = plan_pool()
if plan is None and ALLOW_HERMES_GLAIVE_EXPANSION:
    print("Primary pool cannot satisfy the grouped count/length policy; auditing the pinned Hermes Glaive subset.")
    audit_source(EXPANSION)
    items, dedup_exclusions, retention, plan = plan_pool()

write_json(AUDIT / "retention.json", retention)
write_jsonl(AUDIT / "dedup-exclusions.jsonl", dedup_exclusions)
write_json(AUDIT / "pool-summary.json", {
    "source_files": source_manifest, "strict_candidates_before_dedup": len(source_candidates),
    "unique_schema_valid_rows": len(items), "components": len({x["group_id"] for x in items}),
    "largest_component": max(collections.Counter(x["group_id"] for x in items).values(), default=0),
    "retention": retention,
})
print("Retention / split feasibility:")
for row in retention:
    print(row)
assert plan is not None, (
    f"DATA FEASIBILITY BLOCKER: cannot obtain {TRAIN_MIN} training + {VAL_TARGET} validation + "
    f"{TEST_TARGET} test rows under the frozen grouping/length policy. Inspect audit files; "
    "do not pad, repair labels, split a family, or silently shrink the requirement."
)
MAX_LENGTH, splits, group_assignments = plan
assert TRAIN_MIN <= len(splits["train"]) <= TRAIN_TARGET
assert len(splits["validation"]) == VAL_TARGET and len(splits["test"]) == TEST_TARGET
print("Selected maximum length:", MAX_LENGTH)
print("Split counts:", {name: len(rows) for name, rows in splits.items()})

Deduplication, conflicting-reference, transitive grouping and deterministic split fixtures passed.


func-calling-singleturn.json: reconstructing file:   0%|          |  0.00B / 17.8MB            

func-calling-singleturn.json: downloading bytes:           |  0.00B            

func-calling-singleturn.json {'source_rows': 1893, 'schema_valid_call_rows': 1030, 'excluded': 863}
Audited exact lengths/masks for 500/1030 rows
Audited exact lengths/masks for 1000/1030 rows
Primary pool cannot satisfy the grouped count/length policy; auditing the pinned Hermes Glaive subset.


glaive-function-calling-5k.json: reconstructing file:   0%|          |  0.00B / 20.4MB            

glaive-function-calling-5k.json: downloading bytes:           |  0.00B            

glaive-function-calling-5k.json {'source_rows': 5209, 'schema_valid_call_rows': 3207, 'excluded': 2002}
Audited exact lengths/masks for 500/4218 rows
Audited exact lengths/masks for 1000/4218 rows
Audited exact lengths/masks for 1500/4218 rows
Audited exact lengths/masks for 2000/4218 rows
Audited exact lengths/masks for 2500/4218 rows
Audited exact lengths/masks for 3000/4218 rows
Audited exact lengths/masks for 3500/4218 rows
Audited exact lengths/masks for 4000/4218 rows
Retention / split feasibility:
{'max_length': 512, 'retained_rows': 3248, 'feasible_grouped_split': False}
{'max_length': 1024, 'retained_rows': 4004, 'feasible_grouped_split': True}
{'max_length': 1536, 'retained_rows': 4197, 'feasible_grouped_split': True}
{'max_length': 2048, 'retained_rows': 4218, 'feasible_grouped_split': True}
Selected maximum length: 1024
Split counts: {'train': 1600, 'validation': 150, 'test': 150}


## 7. Prove split isolation and check the actual training collator

These assertions cover IDs, exact normalized requests, tool names, schema
families and components. They do not claim semantic paraphrase detection.
The CPU padding probe uses the pinned TRL collator used by Soup's plain SFT
path, including a `pad_token_id == eos_token_id` case. Real EOT labels must
survive while padded positions stay `-100`.

In [7]:
def isolation_keys(rows):
    keys = set()
    for row in rows:
        keys.add(("component", row["group_id"]))
        keys.add(("request", row["request_key"]))
        keys.add(("input", row["input_key"]))
        keys.update(("source_id", entry["source_id"]) for entry in row["lineage"])
        keys.update(("tool", name.casefold()) for name in row["tool_names"])
        keys.update(("schema", name) for name in row["schema_keys"])
    return keys

split_keys = {name: isolation_keys(rows) for name, rows in splits.items()}
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not (split_keys[a] & split_keys[b]), f"Leakage between {a} and {b}"
assert len({row["id"] for rows in splits.values() for row in rows}) == sum(map(len, splits.values()))
assert all(row["tokens"]["total_tokens"] <= MAX_LENGTH for rows in splits.values() for row in rows)

train_order = sorted(splits["train"], key=lambda row: row["tokens"]["total_tokens"])
examples = [format_row({"messages": row["messages"]}) for row in (train_order[0], train_order[-1])]
assert len(examples[0]["input_ids"]) < len(examples[1]["input_ids"]), "Padding probe needs unequal row lengths."
for pad_id in (tokenizer.pad_token_id, tokenizer.eos_token_id):
    collator = DataCollatorForLanguageModeling(pad_token_id=pad_id, padding_free=False)
    batch = collator(examples)
    for i, example in enumerate(examples):
        n = len(example["input_ids"])
        assert batch["labels"][i, :n].tolist() == example["labels"]
        assert (batch["labels"][i, n:] == -100).all().item()
        assert (batch["attention_mask"][i, n:] == 0).all().item()
        assert any(label == tokenizer.eos_token_id for label in batch["labels"][i, :n].tolist())
write_json(AUDIT / "integrity-checks.json", {
    "cross_split_key_overlap": 0, "all_selected_masks_checked": True,
    "native_template_equivalence_checked": True, "collator": "trl 0.29.1 DataCollatorForLanguageModeling",
    "padding_check_passed": True, "pad_equals_eos_check_passed": True,
    "scope": "structural checks only; semantic reference correctness still requires review",
})
print("Split isolation, exact response/EOT masks and both collator padding checks passed.")

Split isolation, exact response/EOT masks and both collator padding checks passed.


## 8. Export Soup inputs and audit evidence

`train.jsonl`, `validation.jsonl` and `test.jsonl` contain only `messages`, so
metadata is never accidentally inserted into the model prompt. Reference and
lineage information is stored in separate aligned files. The test set is
written but never previewed here. Do not tune from its answers.

`soup-data.yaml` is a **data and masking configuration fragment**, not a command to start
training. It fixes `val_split: 0` because the next notebook must load the
explicit validation file and pass `{"train": ..., "val": ...}` to Soup's
`SFTTrainerWrapper.setup`. Soup's automatic local split must not reassign these
families. The model weights must later be downloaded at the pinned model
revision; the tokenizer folder here contains no model weights.

In [8]:
selected_ids = {row["id"] for rows in splits.values() for row in rows}
selection_exclusions = []
for row in items:
    if row["id"] not in selected_ids:
        selection_exclusions.append({"id": row["id"], "source_file": row["source_file"],
            "source_index": row["source_index"], "reason": "over_selected_length" if row["tokens"]["total_tokens"] > MAX_LENGTH else "not_selected_under_cap",
            "total_tokens": row["tokens"]["total_tokens"], "group_id": row["group_id"],
            "assigned_split": group_assignments.get(row["group_id"])})
write_jsonl(AUDIT / "excluded.jsonl", [*raw_rejections, *dedup_exclusions, *selection_exclusions])

def reference(row):
    return {key: row[key] for key in ("id", "source_id", "source_file", "source_index", "lineage",
            "raw_row_sha256", "group_id", "category", "task", "tools", "calls", "call_count", "tokens", "target_sha256")}

for name, rows in splits.items():
    write_jsonl(PREPARED / f"{name}.jsonl", ({"messages": row["messages"]} for row in rows))
    write_jsonl(PREPARED / f"{name}.references.jsonl", (reference(row) for row in rows))
    write_jsonl(PREPARED / f"{name}.prompts.jsonl", ({"id": row["id"], "messages": row["messages"][:-1]} for row in rows))
    write_jsonl(AUDIT / f"{name}.masks.jsonl", ({"id": row["id"], **format_row({"messages": row["messages"]})} for row in rows))
write_jsonl(AUDIT / "normalization-ledger.jsonl", (
    {"id": row["id"], "source_file": row["source_file"], "source_index": row["source_index"],
     "raw_target": row["raw_target"], "normalized_target": row["messages"][-1]["content"],
     "raw_target_sha256": digest(row["raw_target"]), "normalized_target_sha256": row["target_sha256"],
     "system_and_user_preserved": True}
    for rows in splits.values() for row in rows
))
write_json(AUDIT / "split-manifest.json", {
    "seed": SEED, "max_length": MAX_LENGTH, "policy_sha256": digest(POLICY),
    "group_assignments": group_assignments,
    "selected": {name: [{"id": row["id"], "group_id": row["group_id"], "input_key": row["input_key"],
                         "messages_sha256": digest(row["messages"])} for row in rows] for name, rows in splits.items()},
})

review_sample = [row for name in ("train", "validation") for row in
                 sorted(splits[name], key=lambda x: seeded_key(["semantic-review", x["id"]]))[:20]]
write_jsonl(AUDIT / "semantic-review-sample.jsonl", (
    {"id": row["id"], "messages": row["messages"], "tools": row["tools"],
     "review": "Check that the requested action, selected tools and every argument agree; schema validity alone is insufficient."}
    for row in review_sample
))

data_settings = {"train": "./data/prepared/train.jsonl", "format": "chatml", "val_split": 0,
    "max_length": MAX_LENGTH, "train_on_responses_only": True, "chat_template": STRICT_TEMPLATE}
(OUT / "soup-data.yaml").write_text(yaml.safe_dump({"data": data_settings,
    "training": {"train_on_eot": True, "packing": False}}, sort_keys=False, allow_unicode=True))
write_json(OUT / "training-inputs.json", {
    "dataset": DATA_ID, "dataset_revision": DATA_REV,
    "base_model": MODEL_ID, "model_revision": MODEL_REV, "soup_commit": SOUP_REV,
    "tokenizer_dir": "tokenizer", "data_config_fragment": "soup-data.yaml",
    "train": "data/prepared/train.jsonl", "validation": "data/prepared/validation.jsonl", "test": "data/prepared/test.jsonl",
    "soup_training_settings": {"train_on_eot": True, "packing": False},
    "validation_loading": "Use explicit train/val lists in SFTTrainerWrapper.setup; retain data.val_split=0.",
    "base_loading": "Download the pinned base weights in the training notebook; this bundle only includes tokenizer files.",
    "max_length": MAX_LENGTH, "counts": {name: len(rows) for name, rows in splits.items()},
})

breakdown = {}
for name, rows in splits.items():
    lengths = sorted(row["tokens"]["total_tokens"] for row in rows)
    breakdown[name] = {"rows": len(rows), "components": len({row["group_id"] for row in rows}),
        "sources": dict(collections.Counter(row["source_file"] for row in rows)),
        "call_counts": dict(collections.Counter(str(row["call_count"]) for row in rows)),
        "categories": dict(collections.Counter(str(row["category"]) for row in rows)),
        "tokens": {"min": lengths[0], "median": lengths[len(lengths)//2], "p95": lengths[math.ceil(0.95*len(lengths))-1], "max": lengths[-1]},
        "shifted_supervised_tokens": sum(row["tokens"]["shifted_supervised_tokens"] for row in rows)}
write_json(AUDIT / "data-audit.json", {
    "sources": source_manifest, "counts_and_distribution": breakdown,
    "raw_exclusion_reasons": dict(collections.Counter(row["reason"] for row in raw_rejections)),
    "dedup_exclusion_reasons": dict(collections.Counter(row["reason"] for row in dedup_exclusions)),
    "selection_exclusion_reasons": dict(collections.Counter(row["reason"] for row in selection_exclusions)),
    "retention": retention, "semantic_review_status": "sample exported; not automatically verified",
    "known_limits": ["Structural grouping does not detect every paraphrase.",
                     "Holdout families are selected from smaller components.",
                     "Only call-only first turns are covered; clarification/abstention and multi-turn behavior are outside this policy."],
})
print(json.dumps({name: {k: v for k, v in details.items() if k != "categories"} for name, details in breakdown.items()}, indent=2))
print("Soup data settings exported; no training has started.")

{
  "train": {
    "rows": 1600,
    "components": 385,
    "sources": {
      "func-calling-singleturn.json": 519,
      "glaive-function-calling-5k.json": 1081
    },
    "call_counts": {
      "1": 1333,
      "3": 121,
      "2": 127,
      "4": 15,
      "5": 2,
      "9": 2
    },
    "tokens": {
      "min": 307,
      "median": 424,
      "p95": 918,
      "max": 1022
    },
    "shifted_supervised_tokens": 63502
  },
  "validation": {
    "rows": 150,
    "components": 150,
    "sources": {
      "func-calling-singleturn.json": 149,
      "glaive-function-calling-5k.json": 1
    },
    "call_counts": {
      "3": 42,
      "1": 69,
      "2": 34,
      "5": 1,
      "4": 4
    },
    "tokens": {
      "min": 401,
      "median": 753,
      "p95": 993,
      "max": 1024
    },
    "shifted_supervised_tokens": 11761
  },
  "test": {
    "rows": 150,
    "components": 150,
    "sources": {
      "func-calling-singleturn.json": 148,
      "glaive-function-calling-5k.json": 2
    }

## 9. Run Soup's own data checks

Every split must be 100% format-valid, and data doctor scans **every selected
example** using the saved marked tokenizer, response-only labels and EOT.
It must return `OK`; otherwise the ready bundle is not created. Only training
examples receive displayed mask previews. Full raw CLI output goes to files
to avoid the bulk-output stall seen in the earlier smoke attempt.

`data stats` token estimates are advisory. The exact Qwen/Soup token counts
above control length selection. Environment doctor and `pip check` remain
advisory here because unrelated optional/shared-image packages may warn.

In [10]:
def data_doctor_args(name, path, report_path, row_count):
    args = ["soup", "data", "doctor", str(path), "--model", str(OUT / "tokenizer"),
            "--format", "chatml", "--max-length", str(MAX_LENGTH), "--sample", str(row_count),
            "--train-on-eot", "--output", str(report_path)]
    # Zero is invalid in Soup; omission disables previews without disabling checks.
    if name == "train":
        args.extend(["--show-mask", "3"])
    return args

run_command(["soup", "doctor"], "soup-doctor", check=False, timeout=180)
run_command(["soup", "env", "lock", "--output", str(AUDIT / "soup-env.lock")], "soup-env-lock", check=False, timeout=180)
run_command(["soup", "data", "inspect", str(PREPARED / "train.jsonl"), "--rows", "2"], "train-inspect", timeout=180)
run_command(["soup", "data", "stats", str(PREPARED / "train.jsonl")], "train-stats", timeout=180)
soup_reports = {}
for name, rows in splits.items():
    path = PREPARED / f"{name}.jsonl"
    run_command(["soup", "data", "validate", str(path), "--format", "chatml", "--min-valid-fraction", "1"],
                f"{name}-validate", timeout=180)
    report_path = AUDIT / f"{name}.data-doctor.json"
    run_command(data_doctor_args(name, path, report_path, len(rows)),
                f"{name}-data-doctor", timeout=600)
    report = json.loads(report_path.read_text())
    assert report["rows_scanned"] == len(rows), f"{name}: doctor did not scan all rows"
    assert report["overall"] == "OK", f"{name}: inspect {report_path} before training"
    soup_reports[name] = {"overall": report["overall"], "rows_scanned": report["rows_scanned"]}
print("Soup data checks:", soup_reports)

soup-doctor: exit=0; 63 log lines → audit/logs/soup-doctor.log
soup-env-lock: exit=0; 5 log lines → audit/logs/soup-env-lock.log
train-inspect: exit=0; 25 log lines → audit/logs/train-inspect.log
train-stats: exit=0; 40 log lines → audit/logs/train-stats.log
train-validate: exit=0; 3 log lines → audit/logs/train-validate.log
train-data-doctor: exit=0; 167 log lines → audit/logs/train-data-doctor.log
validation-validate: exit=0; 3 log lines → audit/logs/validation-validate.log
validation-data-doctor: exit=0; 42 log lines → audit/logs/validation-data-doctor.log
test-validate: exit=0; 3 log lines → audit/logs/test-validate.log
test-data-doctor: exit=0; 42 log lines → audit/logs/test-data-doctor.log
Soup data checks: {'train': {'overall': 'OK', 'rows_scanned': 1600}, 'validation': {'overall': 'OK', 'rows_scanned': 150}, 'test': {'overall': 'OK', 'rows_scanned': 150}}


## 10. Freeze and download the training-data bundle

The ready marker means the **automated preparation checks** passed. It is not
a model-quality result or a SHIP verdict. Review `semantic-review-sample.jsonl`
before the full experiment. Raw source files remain in the working directory;
the ZIP includes their hashes, pinned download identities and dataset card
instead of duplicating large source files.

Download the ZIP from Kaggle's **Output** panel (or the link below), then
attach it as a private dataset/input to the training notebook. Do not change
splits, targets, tokenizer/template or sequence length after freezing a run
without creating and recording a new preparation run.

In [11]:
(OUT / "README.md").write_text(
    "# Prepared Hermes data for Soup\n\n"
    f"Dataset: {DATA_ID}@{DATA_REV}\nModel/tokenizer: {MODEL_ID}@{MODEL_REV}\nSoup: {SOUP_REV}\n\n"
    f"Counts: { {name: len(rows) for name, rows in splits.items()} }; max_length={MAX_LENGTH}.\n\n"
    "Merge soup-data.yaml into your training config. Keep response-only labels, the exported raw Jinja template, "
    "train_on_eot=true, packing=false, and val_split=0. Load explicit train/validation lists through Soup's wrapper. "
    "Do not use a directory glob that could include test.jsonl. Download pinned model weights separately.\n\n"
    "The tokenizer directory has the marked template and no model weights. Test prompts have no answers; "
    "test references are reserved for final evaluation. Preserve the audit, exclusions, split manifest and hashes.\n\n"
    "Automatic structural checks do not establish semantic reference correctness or GPU fit at untested lengths. "
    "Review audit/semantic-review-sample.jsonl and calibrate the T4 training setup before a full run.\n"
)
(AUDIT / "dataset-card.md").write_bytes((RAW / "README.md").read_bytes())
write_json(OUT / "READY.json", {
    "automated_preparation_passed": True, "run_id": RUN_ID,
    "finished_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "counts": {name: len(rows) for name, rows in splits.items()},
    "max_length": MAX_LENGTH, "soup_data_doctor": soup_reports,
    "semantic_review": "Required human review is outside automated structural checks; sample supplied.",
    "training_ran": False, "model_weights_downloaded": False,
})

package_files = sorted(
    [p for folder in (PREPARED, AUDIT, OUT / "tokenizer") for p in folder.rglob("*") if p.is_file()]
    + [OUT / name for name in ("README.md", "READY.json", "training-inputs.json", "soup-data.yaml", "chat-template.jinja")]
)
manifest = {str(path.relative_to(OUT)): {"sha256": file_sha(path), "bytes": path.stat().st_size} for path in package_files}
write_json(OUT / "manifest.json", manifest)
package_files.append(OUT / "manifest.json")
bundle = OUT.parent / f"hermes-soup-data-{RUN_ID}.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in package_files:
        archive.write(path, arcname=str(Path("hermes-soup-data") / path.relative_to(OUT)))
with zipfile.ZipFile(bundle) as archive:
    assert archive.testzip() is None
    assert len(archive.namelist()) == len(set(archive.namelist())) == len(package_files)
    for relative, expected in manifest.items():
        assert hashlib.sha256(archive.read("hermes-soup-data/" + relative)).hexdigest() == expected["sha256"]

print("AUTOMATED DATA PREPARATION: PASS")
print("Bundle:", bundle)
print("Bundle SHA-256:", file_sha(bundle))
print("No training, model evaluation or GPU-fit claim has been made.")
from IPython.display import FileLink, display
display(FileLink(str(bundle)))

AUTOMATED DATA PREPARATION: PASS
Bundle: /kaggle/working/hermes-soup-data/hermes-soup-data-20261109T210817498784Z.zip
Bundle SHA-256: 178702362546333ae34ba31be7d8e64504218141f18deec2eb9139b72b62792a
No training, model evaluation or GPU-fit claim has been made.


/kaggle/working/hermes-soup-data/hermes-soup-data-20261109T210817498784Z.zip

## References and next step

- [Pinned Hermes dataset card](https://huggingface.co/datasets/NousResearch/hermes-function-calling-v1/blob/dae3e1d28cfbcf4b915c04ea1e072030529b4bda/README.md)
- [Pinned Qwen tokenizer configuration](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/blob/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json)
- [Pinned Soup formatter](https://github.com/MakazhanAlpamys/Soup/blob/4cff43bf967e96a484381205e5125e2fc39227f1/src/soup_cli/data/sft_format.py)
- [Pinned Soup loss masking](https://github.com/MakazhanAlpamys/Soup/blob/4cff43bf967e96a484381205e5125e2fc39227f1/src/soup_cli/data/loss_mask.py)
- [Pinned TRL collator](https://github.com/huggingface/trl/blob/v0.29.1/trl/trainer/sft_trainer.py)

The next notebook should load this immutable bundle, establish the analytical
memory estimate, calibrate batch/length on one T4, and then run the full Soup
training and B/A/M/Q verification. Keep this preparation notebook separate so
the data audit can be reproduced without spending GPU quota.